# 1)Import Necessary Libraries

In [89]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

# 2)Import Dataset

In [43]:
weather_pred=pd.read_csv("environmental_data.csv")
weather_pred

,Unnamed: 0,Ozone,Solar.R,Wind,Temp C,Month,Day,Year,Temp,Weather
0,1,41.0,190.0,7.4,67,5,1,2010,67,S
1,2,36.0,118.0,8.0,72,5,2,2010,72,C
2,3,12.0,149.0,12.6,74,5,3,2010,74,PS
3,4,18.0,313.0,11.5,62,5,4,2010,62,S
4,5,NaN,NaN,14.3,56,5,5,2010,56,S
...,...,...,...,...,...,...,...,...,...,...
153,154,41.0,190.0,7.4,67,5,1,2010,67,C
154,155,30.0,193.0,6.9,70,9,26,2010,70,PS
155,156,NaN,145.0,13.2,77,9,27,2010,77,S
156,157,14.0,191.0,14.3,75,9,28,2010,75,S


# 3)Data Understanding

## 3.1 perform initial analysis on the Data

In [44]:
weather_pred.shape

(158, 10)

In [45]:
weather_pred.describe()

,Unnamed: 0,Ozone,Solar.R,Wind,Day,Year,Temp
count,158.000000,120.000000,151.000000,158.000000,158.000000,158.0,158.000000
mean,79.500000,41.583333,185.403974,9.957595,16.006329,2010.0,77.727848
std,45.754781,32.620709,88.723103,3.511261,8.997166,0.0,9.377877
min,1.000000,1.000000,7.000000,1.700000,1.000000,2010.0,56.000000
25%,40.250000,18.000000,119.000000,7.400000,8.000000,2010.0,72.000000
50%,79.500000,30.500000,197.000000,9.700000,16.000000,2010.0,78.500000
75%,118.750000,61.500000,257.000000,11.875000,24.000000,2010.0,84.000000
max,158.000000,168.000000,334.000000,20.700000,31.000000,2010.0,97.000000


In [46]:
weather_pred.dtypes

Unnamed: 0      int64
Ozone         float64
Solar.R       float64
Wind          float64
Temp C         object
Month          object
Day             int64
Year            int64
Temp            int64
Weather        object
dtype: object

In [47]:
weather_pred["Weather"].unique()

array(['S', 'C', 'PS', nan], dtype=object)

In [48]:
weather_pred["Month"].unique()

array(['5', 'May', '6', '7', '8', '9'], dtype=object)

In [49]:
weather_pred["Weather"].value_counts(dropna=False)

Weather
S      59
C      49
PS     47
NaN     3
Name: count, dtype: int64

# 4)Data preparation

##  Stage 1 - Data Cleaning - drop, fillna with mean, median, mode Imputation, changing dtype

In [50]:
weather_pred.drop(labels = ["Unnamed: 0","Temp C"],axis = 1,inplace=True)


In [34]:
weather_pred.head()


,Ozone,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,41.0,190.0,7.4,5,1,2010,67,S
1,36.0,118.0,8.0,5,2,2010,72,C
2,12.0,149.0,12.6,5,3,2010,74,PS
3,18.0,313.0,11.5,5,4,2010,62,S
4,NaN,NaN,14.3,5,5,2010,56,S


### Note 1:We removed unwanted columns - UnNamed 0, and Temp C.

In [51]:
weather_pred.isna().sum()

Ozone      38
Solar.R     7
Wind        0
Month       0
Day         0
Year        0
Temp        0
Weather     3
dtype: int64

In [52]:
38/158

0.24050632911392406

In [53]:
del weather_pred["Ozone"]

###  Note 2:I'm dropping this column, as 25% of the data is NaN.


In [54]:
weather_pred.isna().sum()

Solar.R    7
Wind       0
Month      0
Day        0
Year       0
Temp       0
Weather    3
dtype: int64

In [55]:
7/158

0.04430379746835443

In [57]:
weather_pred[weather_pred["Solar.R"].isna()]

,Solar.R,Wind,Month,Day,Year,Temp,Weather
4,NaN,14.3,5,5,2010,56,S
5,NaN,14.9,5,6,2010,66,C
10,NaN,6.9,5,11,2010,74,C
26,NaN,8.0,5,27,2010,57,PS
95,NaN,6.9,8,4,2010,86,NaN
96,NaN,7.4,8,5,2010,85,S
97,NaN,4.6,8,6,2010,87,C


In [59]:
weather_pred.groupby(by="Weather")["Solar.R"].mean().round(1)

Weather
C     192.4
PS    174.3
S     193.3
Name: Solar.R, dtype: float64

In [62]:
weather_pred["Solar.R"] = weather_pred.groupby("Weather")["Solar.R"].transform(lambda x:x.fillna(x.mean().round()))
weather_pred

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,C
154,193.0,6.9,9,26,2010,70,PS
155,145.0,13.2,9,27,2010,77,S
156,191.0,14.3,9,28,2010,75,S


In [64]:
weather_pred.isna().sum()

Solar.R    3
Wind       0
Month      0
Day        0
Year       0
Temp       0
Weather    3
dtype: int64

In [68]:
weather_pred[weather_pred["Solar.R"].isna()]

,Solar.R,Wind,Month,Day,Year,Temp,Weather
87,NaN,12.0,7,27,2010,86,NaN
93,NaN,13.8,8,2,2010,81,NaN
95,NaN,6.9,8,4,2010,86,NaN


In [70]:
weather_pred.dropna(inplace=True)
weather_pred.isna().sum()

Solar.R    0
Wind       0
Month      0
Day        0
Year       0
Temp       0
Weather    0
dtype: int64

In [71]:
weather_pred["Month"].unique()




array(['5', 'May', '6', '7', '8', '9'], dtype=object)

In [78]:
weather_pred["Month"].replace(to_replace="May",value=5,inplace=True)
weather_pred["Month"].unique()

array(['5', 5, '6', '7', '8', '9'], dtype=object)

### Data Type Transformation

In [79]:
weather_pred.dtypes

Solar.R    float64
Wind       float64
Month       object
Day          int64
Year         int64
Temp         int64
Weather     object
dtype: object

In [84]:
weather_pred["Month"]=weather_pred["Month"].astype('int')
weather_pred.dtypes

Solar.R    float64
Wind       float64
Month        int64
Day          int64
Year         int64
Temp         int64
Weather     object
dtype: object

## Stage 2

### 2. Data Transformation
    * If data is CONTINUOUS = Standard Scaler, MinMaxScaler, Robust Scaler
    * If data is DISCRETE   = Label Encoder, One hot Encoder

## How to choose between Label Encoder and One Hot Encoder?

### Input feature:
Based on whether it is **Parametric or a Non Parametric Model**, we will choosing between LE or OHE.
* For **Parametric Models** - better to go with **OHE.**
* For **Non-Parametric Models** - always go with **Label Encoding.**

### Output feature:
Always the output feature, it must be **Label Encoder.**

### 1)Using Label Encoder

In [87]:
weather_pred_copy_1=weather_pred.copy()
weather_pred_copy_1.head()

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S


In [93]:
lab_con=LabelEncoder()
weather_pred_copy_1["Weather"]=lab_con.fit_transform(weather_pred_copy_1["Weather"])
weather_pred_copy_1

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,2
1,118.0,8.0,5,2,2010,72,0
2,149.0,12.6,5,3,2010,74,1
3,313.0,11.5,5,4,2010,62,2
4,193.0,14.3,5,5,2010,56,2
...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,0
154,193.0,6.9,9,26,2010,70,1
155,145.0,13.2,9,27,2010,77,2
156,191.0,14.3,9,28,2010,75,2


In [94]:
weather_pred_copy_1.dtypes

Solar.R    float64
Wind       float64
Month        int64
Day          int64
Year         int64
Temp         int64
Weather      int64
dtype: object

### 2. One Hot Encoder

This can be achieved by using any of the following 2 libraries:

1. Pandas - pd.get_dummies()
2. sklearn - OneHotEncoder()

In [101]:
weather_pred_copy_2=weather_pred.copy()
weather_pred_copy_2.head()

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S


In [102]:
weather_pred_copy_2=pd.get_dummies(data=weather_pred_copy_2,columns=["Weather"],dtype='int')
weather_pred_copy_2

,Solar.R,Wind,Month,Day,Year,Temp,Weather_C,Weather_PS,Weather_S
0,190.0,7.4,5,1,2010,67,0,0,1
1,118.0,8.0,5,2,2010,72,1,0,0
2,149.0,12.6,5,3,2010,74,0,1,0
3,313.0,11.5,5,4,2010,62,0,0,1
4,193.0,14.3,5,5,2010,56,0,0,1
...,...,...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,1,0,0
154,193.0,6.9,9,26,2010,70,0,1,0
155,145.0,13.2,9,27,2010,77,0,0,1
156,191.0,14.3,9,28,2010,75,0,0,1


In [106]:
weather_pred_copy_3=weather_pred.copy()
weather_pred_copy_3.head()

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S


In [109]:
One_con=OneHotEncoder()
weather_pred_copy_3=One_con.fit_transform(weather_pred_copy_3[["Weather"]])
weather_pred_copy_3

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 155 stored elements and shape (155, 3)>

In [111]:
encoded_df = pd.DataFrame(
    weather_pred_copy_3.toarray(),
    columns=One_con.get_feature_names_out(["Weather"])
)

display(encoded_df)

,Weather_C,Weather_PS,Weather_S
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0
...,...,...,...
150,1.0,0.0,0.0
151,0.0,1.0,0.0
152,0.0,0.0,1.0
153,0.0,0.0,1.0


# The End!